# Nifty 50 Option Chain Data Analysis & Visualization
## A Complete Guide to Trading Data Analysis with Pandas and Matplotlib

This notebook covers:
1. Fetching live option chain data
2. Data analysis and manipulation
3. Visualization strategies
4. Put-Call Ratio (PCR) analysis
5. Performance metrics calculation

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from datetime import datetime
from urllib.parse import quote, quote_plus
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All libraries imported successfully!")

## 2. Set Up API Connection

In [ ]:
# Get access token
ACCESS_TOKEN = os.getenv("UPSTOX_ACCESS_TOKEN")

BASE_HEADERS = {
    "Accept": "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

def get_option_chain_data(symbol="NSE_INDEX|Nifty 50", expiry_date="2026-02-17"):
    """Fetch option chain data from Upstox API"""
    url = f"https://api.upstox.com/v2/option/chain?instrument_key={quote(symbol, safe='|')}&expiry_date={quote_plus(str(expiry_date))}"
    url = url.replace(' ', '')
    
    try:
        resp = requests.get(url, headers=BASE_HEADERS, timeout=10)
        if resp.status_code == 200:
            return resp.json().get('data', [])
        else:
            print(f"Error: Status {resp.status_code}")
            return []
    except Exception as e:
        print(f"Error fetching data: {e}")
        return []

def get_spot_price(symbol="NSE_INDEX|Nifty 50"):
    """Get current spot price"""
    url = f"https://api.upstox.com/v3/market-quote/ltp?instrument_key={quote(symbol, safe='|,:')}"
    
    try:
        resp = requests.get(url, headers=BASE_HEADERS, timeout=10)
        if resp.status_code == 200:
            data = resp.json().get('data', {})
            return data.get(symbol, {}).get('last_price', 0)
        return 0
    except:
        return 0

print("✓ API functions defined!")

## 3. Fetch and Prepare Data

In [ ]:
# Fetch data
option_chain = get_option_chain_data()
spot_price = get_spot_price()

print(f"Current Nifty 50: {spot_price}")
print(f"Total strikes in chain: {len(option_chain)}")

# Prepare data for analysis
data = []
for strike_data in option_chain:
    ce = strike_data.get('call_options', {}) or {}
    pe = strike_data.get('put_options', {}) or {}
    
    ce_market = ce.get('market_data', {}) or {}
    pe_market = pe.get('market_data', {}) or {}
    
    strike = strike_data.get('strike_price', 0)
    
    ce_oi = ce_market.get('oi', 0)
    pe_oi = pe_market.get('oi', 0)
    pcr = pe_oi / ce_oi if ce_oi > 0 else 0
    
    data.append({
        'Strike': strike,
        'Call_LTP': ce_market.get('ltp', 0),
        'Call_OI': ce_oi,
        'Call_IV': ce.get('option_greeks', {}).get('iv', 0),
        'Call_Delta': ce.get('option_greeks', {}).get('delta', 0),
        'Put_LTP': pe_market.get('ltp', 0),
        'Put_OI': pe_oi,
        'Put_IV': pe.get('option_greeks', {}).get('iv', 0),
        'Put_Delta': pe.get('option_greeks', {}).get('delta', 0),
        'PCR': pcr,
        'Total_OI': ce_oi + pe_oi,
        'Spot_Distance': abs(strike - spot_price)
    })

# Create DataFrame
df = pd.DataFrame(data)
df = df.sort_values('Strike').reset_index(drop=True)

print(f"\n✓ Data prepared! Shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

## 4. Data Analysis - Key Metrics

In [ ]:
# Find ATM (At-The-Money) strike
atm_idx = (df['Strike'] - spot_price).abs().idxmin()
atm_strike = df.loc[atm_idx, 'Strike']

# Key metrics
total_call_oi = df['Call_OI'].sum()
total_put_oi = df['Put_OI'].sum()
market_pcr = total_put_oi / total_call_oi
atm_pcr = df.loc[atm_idx, 'PCR']

# Max pain analysis
max_oi_strike_idx = df['Total_OI'].idxmax()
max_oi_strike = df.loc[max_oi_strike_idx, 'Strike']

print(f"{'='*50}")
print(f"OPTION CHAIN ANALYSIS SUMMARY")
print(f"{'='*50}")
print(f"Current Spot Price: {spot_price:.2f}")
print(f"ATM Strike: {atm_strike:.0f}")
print(f"\nOPEN INTEREST ANALYSIS:")
print(f"  Total Call OI: {total_call_oi:,.0f}")
print(f"  Total Put OI: {total_put_oi:,.0f}")
print(f"  Market PCR: {market_pcr:.2f}")
print(f"  ATM PCR: {atm_pcr:.2f}")
print(f"\nRISK LEVELS:")
print(f"  Max Pain Strike: {max_oi_strike:.0f}")
print(f"  Max OI Value: {df['Total_OI'].max():,.0f}")
print(f"\nVOLATILITY ANALYSIS:")
print(f"  Avg Call IV: {df['Call_IV'].mean():.2f}")
print(f"  Avg Put IV: {df['Put_IV'].mean():.2f}")
print(f"{'='*50}")

## 5. Visualization 1: Open Interest Distribution

In [ ]:
# Select ATM +/- 10 strikes for clarity
atm_range = df[(df['Strike'] >= atm_strike - 500) & (df['Strike'] <= atm_strike + 500)].copy()

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(atm_range))
width = 0.35

bars1 = ax.bar(x - width/2, atm_range['Call_OI']/100000, width, label='Call OI', color='green', alpha=0.8)
bars2 = ax.bar(x + width/2, atm_range['Put_OI']/100000, width, label='Put OI', color='red', alpha=0.8)

# Mark ATM strike
atm_x = (atm_range['Strike'] == atm_strike).idxmax()
ax.axvline(x=atm_x - atm_range.index[0], color='blue', linestyle='--', linewidth=2, label=f'ATM ({atm_strike:.0f})')

ax.set_xlabel('Strike Price', fontsize=12, fontweight='bold')
ax.set_ylabel('Open Interest (Lakhs)', fontsize=12, fontweight='bold')
ax.set_title('Open Interest Distribution - Call vs Put', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{s:.0f}' for s in atm_range['Strike']], rotation=45)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/workspaces/Honey/oi_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Open Interest chart saved!")

## 6. Visualization 2: Put-Call Ratio Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# PCR by strike
ax1.plot(atm_range['Strike'], atm_range['PCR'], marker='o', linewidth=2, markersize=6, color='purple')
ax1.axvline(x=atm_strike, color='blue', linestyle='--', alpha=0.7, label='ATM')
ax1.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5, label='PCR = 1.0 (Neutral)')
ax1.set_xlabel('Strike Price', fontsize=11, fontweight='bold')
ax1.set_ylabel('Put-Call Ratio (PCR)', fontsize=11, fontweight='bold')
ax1.set_title('PCR Across Strikes', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# PCR interpretation
colors = ['green' if pcr < 1 else 'red' for pcr in atm_range['PCR']]
ax2.bar(range(len(atm_range)), atm_range['PCR'], color=colors, alpha=0.7)
ax2.axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Neutral (PCR=1)')
ax2.set_xlabel('Strike Index (ATM centered)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Put-Call Ratio', fontsize=11, fontweight='bold')
ax2.set_title('PCR > 1: Bullish | PCR < 1: Bearish', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend()

plt.tight_layout()
plt.savefig('/workspaces/Honey/pcr_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ PCR analysis chart saved!")

## 7. Visualization 3: Price and Greeks Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Call Delta
axes[0, 0].plot(atm_range['Strike'], atm_range['Call_Delta'], marker='o', label='Call Delta', color='green')
axes[0, 0].axvline(x=atm_strike, color='blue', linestyle='--', alpha=0.7)
axes[0, 0].set_title('Call Delta by Strike', fontweight='bold')
axes[0, 0].set_ylabel('Delta')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Put Delta
axes[0, 1].plot(atm_range['Strike'], atm_range['Put_Delta'], marker='o', label='Put Delta', color='red')
axes[0, 1].axvline(x=atm_strike, color='blue', linestyle='--', alpha=0.7)
axes[0, 1].set_title('Put Delta by Strike', fontweight='bold')
axes[0, 1].set_ylabel('Delta')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Call Price
axes[1, 0].plot(atm_range['Strike'], atm_range['Call_LTP'], marker='s', label='Call LTP', color='darkgreen', linewidth=2)
axes[1, 0].axvline(x=atm_strike, color='blue', linestyle='--', alpha=0.7)
axes[1, 0].set_title('Call Option Price by Strike', fontweight='bold')
axes[1, 0].set_ylabel('Price')
axes[1, 0].set_xlabel('Strike')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Put Price
axes[1, 1].plot(atm_range['Strike'], atm_range['Put_LTP'], marker='s', label='Put LTP', color='darkred', linewidth=2)
axes[1, 1].axvline(x=atm_strike, color='blue', linestyle='--', alpha=0.7)
axes[1, 1].set_title('Put Option Price by Strike', fontweight='bold')
axes[1, 1].set_ylabel('Price')
axes[1, 1].set_xlabel('Strike')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('/workspaces/Honey/greeks_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Greeks analysis chart saved!")

## 8. Trading Signal Generation

In [ ]:
# Generate trading signals based on PCR
atm_data = df.loc[atm_idx]

print(f"\n{'='*60}")
print(f"TRADING SIGNAL GENERATION")
print(f"{'='*60}")

# Signal 1: PCR-based sentiment
if market_pcr > 1.2:
    signal1 = "🟢 BULLISH - High Put OI suggests upside confidence"
elif market_pcr < 0.8:
    signal1 = "🔴 BEARISH - Low Put OI suggests downside risk"
else:
    signal1 = "🟡 NEUTRAL - Balanced sentiment"

print(f"\n1. PCR SIGNAL (Market PCR: {market_pcr:.2f})")
print(f"   {signal1}")

# Signal 2: OI Concentration
oi_concentration = df.loc[max_oi_strike_idx, 'Total_OI'] / df['Total_OI'].sum()
if oi_concentration > 0.05:  # More than 5% on one strike
    signal2 = f"⚠️  HIGH OI CONCENTRATION at {max_oi_strike:.0f} ({oi_concentration*100:.1f}%) - Strong support/resistance"
else:
    signal2 = f"✓ OI well distributed - Normal market condition"

print(f"\n2. OI CONCENTRATION SIGNAL")
print(f"   {signal2}")

# Signal 3: Implied Volatility
avg_iv = (df['Call_IV'].mean() + df['Put_IV'].mean()) / 2
if avg_iv > 100:
    signal3 = f"📈 HIGH VOLATILITY ({avg_iv:.1f}%) - Expect larger price swings"
elif avg_iv < 50:
    signal3 = f"📉 LOW VOLATILITY ({avg_iv:.1f}%) - Range-bound market"
else:
    signal3 = f"📊 NORMAL VOLATILITY ({avg_iv:.1f}%) - Standard conditions"

print(f"\n3. VOLATILITY SIGNAL")
print(f"   {signal3}")

# Signal 4: ATM Options Pricing
intrinsic_call = max(0, spot_price - atm_strike)
time_value_call = atm_data['Call_LTP'] - intrinsic_call

print(f"\n4. ATM PRICING ANALYSIS")
print(f"   Call Intrinsic Value: {intrinsic_call:.2f}")
print(f"   Call Time Value: {time_value_call:.2f}")
print(f"   Call LTP: {atm_data['Call_LTP']:.2f}")

print(f"\n{'='*60}")

## 9. Summary Statistics

In [ ]:
# Create summary dataframe
summary = pd.DataFrame({
    'Metric': [
        'Current Spot',
        'ATM Strike',
        'Max Pain Strike',
        'Total Call OI',
        'Total Put OI',
        'Market PCR',
        'ATM PCR',
        'Avg Call IV',
        'Avg Put IV',
        'Total Instruments',
        'Timestamp'
    ],
    'Value': [
        f"{spot_price:.2f}",
        f"{atm_strike:.0f}",
        f"{max_oi_strike:.0f}",
        f"{total_call_oi:,.0f}",
        f"{total_put_oi:,.0f}",
        f"{market_pcr:.2f}",
        f"{atm_pcr:.2f}",
        f"{df['Call_IV'].mean():.2f}%",
        f"{df['Put_IV'].mean():.2f}%",
        f"{len(df)}",
        datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    ]
})

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(summary.to_string(index=False))
print("="*60)

## 10. Next Steps

### Key Learnings:
1. **Open Interest (OI)**: Shows market participation. Higher OI = more liquidity  
2. **Put-Call Ratio (PCR)**: > 1 = Bullish, < 1 = Bearish, ~1 = Neutral  
3. **Greeks**: Delta, Gamma, Vega, Theta help predict option price movements  
4. **Implied Volatility**: Higher IV = Expectation of larger price moves  
5. **Max Pain**: Strike with highest OI concentration often acts as support/resistance  

### Practice Exercises:
- [ ] Calculate your trading P&L using Delta hedging  
- [ ] Build a moving average PCR indicator  
- [ ] Create a volatility smile visualization  
- [ ] Backtest a PCR-based trading strategy  
- [ ] Set up alerts for extreme OI concentrations  

### Resources:
- Pandas Documentation: https://pandas.pydata.org/docs/  
- Matplotlib Guide: https://matplotlib.org/  
- Options Trading: https://www.investopedia.com/options-basics/  
- Upstox API Docs: https://developer.upstox.com/  